In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

from churn.config import BRONZE_TABLE

## 1. Load CSV from Bronze

In [0]:
pdf = spark.table(BRONZE_TABLE).toPandas()

## 2. Basic exploration

In [0]:
pdf.head(10)

In [0]:
pdf.shape

In [0]:
pdf.info()

In [0]:
pdf.describe()

In [0]:
pdf.describe(include="object").T

In [0]:
pdf.value_counts()

In [0]:
print(pdf["OnlineSecurity"].value_counts())
print(pdf["OnlineBackup"].value_counts())
print(pdf["DeviceProtection"].value_counts())
print(pdf["TechSupport"].value_counts())
print(pdf["StreamingTV"].value_counts())
print(pdf["StreamingMovies"].value_counts())

# SPANISH:

## Sobre los datos

Dataset: IBM Telco Customer Churn. Una fila por cliente de una empresa de
telecomunicaciones. La variable objetivo (`Churn`) indica si el cliente se dio
de baja durante el último mes.

## Columnas

- `customerID` — Código único del cliente. No es una variable predictiva. Se excluye del modelo.
- `gender` — Female / Male.
- `SeniorCitizen` — 1 si el cliente tiene 65 años o más, 0 en caso contrario. Categórica binaria.
- `Partner` — Si tiene pareja. Yes / No.
- `Dependents` — Si tiene personas a su cargo. Yes / No.
- `tenure` — Meses que lleva como cliente. Tiene valores en 0.
- `Contract` — Tipo de contrato: Month-to-month / One year / Two year.
- `PaymentMethod` — Forma de pago: Electronic check / Mailed check / Bank transfer (automatic) / Credit card (automatic).
- `PaperlessBilling` — Factura electrónica. Yes / No.
- `PhoneService` — Si tiene línea telefónica. Yes / No.
- `MultipleLines` — Si tiene varias líneas. Yes / No / No phone service.
- `InternetService` — Tipo de conexión: DSL / Fiber optic / No.
- `OnlineSecurity` — Servicio de seguridad contratado. No / Yes / No internet service
- `OnlineBackup` — Copia de seguridad en la nube. No / Yes / No internet service
- `DeviceProtection` — Protección de dispositivos. No / Yes / No internet service
- `TechSupport` — Soporte técnico premium. No / Yes / No internet service
- `StreamingTV` — Televisión en streaming. No / Yes / No internet service
- `StreamingMovies` — Películas en streaming. No / Yes / No internet service
- `MonthlyCharges` — Importe facturado en el último mes.
- `TotalCharges` — Importe total facturado.
- `Churn` — Si el cliente se dio de baja el último mes. Yes / No. **TARGET.**

## Primeras observaciones

**Tipos**

- Numéricas reales: solo `tenure` y `MonthlyCharges`.
- `SeniorCitizen` es numérica pero funciona como categórica binaria.
- `TotalCharges` llega como texto pese a ser un importe. Se investiga aparte.
- Las 18 restantes son categóricas de texto.
- Nulos: ninguno en las 21 columnas.

**Numéricas**

- `tenure`: rango de 0 a 72 meses, media de 32.
- `MonthlyCharges`: rango de 18,25 a 118,75, media de 64,76.
- A destacar: `tenure` está algo sesgada a la derecha — la media (32) es mayor que la mediana (29). Y su mínimo es 0, lo que apunta a clientes recién dados de alta.

**Categóricas**

- La mayoría tiene 2 ó 3 valores distintos.
- `Churn` está desbalanceada: 5174 "No" frente a 1869 "Yes" (26,5 % de bajas).
- A destacar: (pendiente de revisar los valores de las seis columnas de servicios)


# ENGLISH:

## About the data

Dataset: IBM Telco Customer Churn. One row per customer of a telecommunications
company. The target variable (`Churn`) indicates whether the customer left
during the last month.

## Columns

- `customerID` — Unique customer code. Not a predictive variable. Excluded from the model.
- `gender` — Female / Male.
- `SeniorCitizen` — 1 if the customer is 65 or older, 0 otherwise. Binary categorical.
- `Partner` — Whether they have a partner. Yes / No.
- `Dependents` — Whether they have dependents. Yes / No.
- `tenure` — Months as a customer. Contains zeros.
- `Contract` — Contract type: Month-to-month / One year / Two year.
- `PaymentMethod` — Payment method: Electronic check / Mailed check / Bank transfer (automatic) / Credit card (automatic).
- `PaperlessBilling` — Paperless billing. Yes / No.
- `PhoneService` — Whether they have a phone line. Yes / No.
- `MultipleLines` — Whether they have multiple lines. Yes / No / No phone service.
- `InternetService` — Connection type: DSL / Fiber optic / No.
- `OnlineSecurity` — Online security add-on. No / Yes / No internet service
- `OnlineBackup` — Cloud backup. No / Yes / No internet service
- `DeviceProtection` — Device protection. No / Yes / No internet service
- `TechSupport` — Premium tech support. No / Yes / No internet service
- `StreamingTV` — Streaming TV. No / Yes / No internet service
- `StreamingMovies` — Streaming movies. No / Yes / No internet service
- `MonthlyCharges` — Amount billed in the last month.
- `TotalCharges` — Total amount billed to date.
- `Churn` — Whether the customer left during the last month. Yes / No. **Target feature.**

## First observations

**Types**

- Truly numeric: only `tenure` and `MonthlyCharges`.
- `SeniorCitizen` is numeric but behaves as a binary categorical.
- `TotalCharges` arrives as text despite being a monetary amount. Investigated separately.
- The remaining 18 are text categoricals.
- Nulls: none across the 21 columns.

**Numeric**

- `tenure`: ranges from 0 to 72 months, mean of 32.
- `MonthlyCharges`: ranges from 18.25 to 118.75, mean of 64.76.
- Worth noting: `tenure` is slightly right-skewed — the mean (32) is higher than the median (29). Its minimum is 0, which points to newly signed-up customers.

**Categorical**

- Most have 2 or 3 distinct values.
- `Churn` is imbalanced: 5,174 "No" against 1,869 "Yes" (26.5% churn rate).
- Worth noting: the six internet service columns share the value `No internet service`, which appears exactly when `InternetService` is `No`. That is redundant information repeated six times, and Phase 3 will have to decide how to handle it when encoding.

## 3. Investigate `TotalCharges`

In [0]:
# SPA: Analizar la columna TotalCharges y por qué está calificada como object
# ENG: Analyze the TotalCharges column and why it is classified as object

pdf["TotalCharges"].value_counts()

In [0]:
# SPA: Parece que hay un elemento vacío en 11 filas.
# ENG: It seems that there is an empty element in 11 rows.
pdf[pdf["TotalCharges"] == " "].shape

In [0]:
vacios = pdf["TotalCharges"] == " "
pdf.loc[vacios].T

In [0]:
# SPA: Todos los TotalCharges vacíos tienen tenure = 0, pero tenemos que ver si todos los tenure = 0 son TotalCharges vacíos.
# ENG: All empty TotalCharges have tenure = 0, but we have to see if all tenure = 0 are empty TotalCharges

pdf.loc[vacios, "tenure"].value_counts()


In [0]:
pd.crosstab(pdf["tenure"] == 0, pdf["TotalCharges"] == " ")

In [0]:
pdf[pdf["tenure"] == 0].shape

In [0]:
pdf.loc[pdf["TotalCharges"] == " ", "Churn"].value_counts()

# SPANISH:
### Corrección sobre los datos:
Los 11 registros con `TotalCharges` vacíos son exactamente los 11 con `tenure = 0` (comprobado con crosstab). Se trata de un dato ausente dada la naturaleza de los datos. A un cliente recién dado de alta no se le ha facturado nada todavía.  

### Decisiones:
- Se conservan las 11 filas
- `TotalCharges = 0` porque es justamente lo que vale ese vacío.
- Se añade la bandera `is_new_customer = (tenure == 0)` para que el modelo pueda distinguir un cero real de un cero por ausencia del historial.
- Ninguna imputación basada en estadísticos se hace antes del split.

# ENGLISH:

### Note on the data

The 11 records with an empty `TotalCharges` are exactly the 11 with `tenure = 0` (verified with a cross-tabulation). This is not missing data but structurally absent data: the value does not exist rather than having been lost. A customer who has just signed up has not been billed anything yet.

### Decisions

- The 11 rows are kept.
- `TotalCharges = 0`, because that is precisely what the blank means.
- A flag `is_new_customer = (tenure == 0)` is added so the model can tell a genuine zero from a zero caused by the absence of billing history.
- No statistic-based imputation is performed before the split.

## 4. Numeric features:

In [0]:
num = pdf[["tenure", "MonthlyCharges"]].copy()

# Este fillna(0) es temporal, solo para ver las estadísticas
# This fillna(0) is temporary, just to see the statistics
num["TotalCharges"] = pd.to_numeric(pdf["TotalCharges"], errors="coerce").fillna(0)

num.describe()

In [0]:
num.hist(bins=40, figsize=(14, 4), layout=(1, 3))
plt.tight_layout();

### 4.1 Tenure

In [0]:
# SPA: Hay que estudiar tenure por ambos extremos para entender por qué la gráfica aparece así. Seguramente en 70+ sea un "cap"
# ENG: tenure has to be studied at both ends to understand why the graph appears like this. Probably at 70+ it is a "cap"
pdf["tenure"].value_counts().sort_index().tail(10)

In [0]:
pdf["tenure"].value_counts().sort_index().head(10)

In [0]:
display(pd.crosstab(pdf["tenure"], pdf["Churn"]).head(10).reset_index())
display(pd.crosstab(pdf["tenure"], pdf["Churn"], normalize="index").head(10).reset_index())

**SPA**: Con un 62% de clientes que han dado de baja en el primer mes, se descarta hipótesis de campaña de captación. El cap del mes 72 queda abierto. Se añadirá una flag más adelante. Con los datos actuales no podemos saber más.  
**ENG**: With 62% of customers who have churned the first month, the hypothesis of a campaign to attract customers is discarded. The cap at 72 months remains open. A flag will be added later. With the current data we cannot know more.

In [0]:
pdf["baja"] = (pdf["Churn"] == "Yes")

pdf.groupby("tenure")["baja"].mean()

In [0]:
tasa_global = pdf["baja"].mean()

pdf["Churn"].value_counts().plot.bar()
plt.title("Tasa global de churn: {:.2f}%".format(tasa_global * 100))
plt.ylabel("Clientes")
plt.xlabel("Churn");

In [0]:
pdf.groupby("tenure")["baja"].mean().plot(figsize=(12, 4))
plt.axhline(tasa_global, color="red", linestyle="--", label=f"Media: {tasa_global:.1%}")
plt.title("Tasa de baja según antigüedad del cliente")
plt.xlabel("Meses como cliente")
plt.ylabel("Tasa de baja")
plt.legend();

In [0]:
grupo = pdf["tenure"].le(3).map({True: "3 meses o menos", False: "Más de 3 meses"})

pdf.groupby(grupo)["baja"].mean().plot.bar(color=["tab:orange", "tab:blue"])
plt.axhline(tasa_global, color="red", linestyle="--", label=f"Media: {tasa_global:.1%}")
plt.title("Tasa de baja: clientes nuevos frente al resto")
plt.ylabel("Tasa de baja")
plt.xticks(rotation=0)
plt.legend();

In [0]:
print(f"{pdf.loc[pdf['tenure'] <= 3, 'baja'].sum() / pdf['baja'].sum():.1%} " f"de todas las bajas vienen de clientes con 3 meses o menos")

# SPA

## Variables numéricas y antigüedad

### Distribuciones

Las tres numéricas tienen formas que no son campanas:

- `tenure` es **bimodal**: picos en ambos extremos. 613 clientes en el mes 1 y
  362 en el mes 72, frente a una media de unos 120 clientes por mes en el
  tramo central.
- `MonthlyCharges` también es bimodal: una concentración fuerte alrededor de
  20 y una joroba ancha entre 70 y 105.
- `TotalCharges` está muy sesgada a la derecha, con la mayoría de clientes
  por debajo de 2.000 y una cola larga hasta 8.685.

El máximo de `TotalCharges` es coherente: 72 meses x 118,75 de cuota máxima
son 8.550, así que 8.685 entra dentro de lo posible.

### Hipótesis: tope en la antigüedad

El mes 72 acumula 362 clientes frente a unos 95 de media entre los meses 63
y 69. 72 meses son exactamente 6 años, lo que sugiere que la ventana de
extracción de datos abarca ese periodo y que todo cliente más antiguo aparece
con el valor máximo.

No es demostrable con los datos disponibles: al cruzarlo con el tipo de
contrato, el 94,8 % de esos clientes tiene contrato de dos años, pero eso es
compatible tanto con el "cap" como con un grupo real de clientes fieles.

**Decisión para Fase 3:** añadir la flag `tenure_max = (tenure == 72)` para
que el modelo pueda distinguir ese grupo, sin afirmar la causa.

### Hallazgo principal: la retención temprana

La tasa global de baja es del **26,5 %** (1.869 de 7.043 clientes).

Al desglosarla por antigüedad, aparece un patrón muy marcado:

| Antigüedad | Tasa de baja |
|---|---|
| 1 mes | 62,0 % |
| 2 meses | 51,7 % |
| 3 meses | 47,0 % |
| 4 meses | 47,2 % |
| 70 meses | 9,2 % |
| 72 meses | 1,7 % |

Agrupando:

- Clientes con **3 meses o menos**: 1.062 personas (15,1 % de la base),
  con una tasa de baja del **56,2 %**.
- Clientes con **más de 3 meses**: 5.981 personas, con una tasa del **21,3 %**.
- **El 31,9 % de todas las bajas se concentra en ese 15,1 % de clientes.**

**Interpretación de negocio.** El problema de esta empresa no está en captar
clientes sino en conservarlos durante el primer trimestre. Casi un tercio de
todas las bajas se produce entre clientes que llevan menos de cuatro meses, y
más de la mitad de los clientes nuevos se marcha antes de cumplir el trimestre.
El riesgo cae de forma sostenida con la antigüedad: quien supera los dos años
apenas se da de baja.

### Pendiente

- Analizar la bimodalidad de `MonthlyCharges`. Hipótesis: los dos grupos
  corresponden a clientes sin internet frente a clientes con fibra.
- Comprobar la relación entre `TotalCharges` y el producto
  `tenure x MonthlyCharges`.
- Decidir si `TotalCharges` necesita transformación. Depende del modelo:
  los árboles son indiferentes, los modelos lineales y de distancia no.

# ENG

## Numeric features and customer tenure

### Distributions

None of the three numeric features is bell-shaped:

- `tenure` is **bimodal**, with peaks at both ends: 613 customers at month 1
  and 362 at month 72, against roughly 120 customers per month in the middle
  range.
- `MonthlyCharges` is also bimodal: a strong cluster around 20 and a wide hump
  between 70 and 105.
- `TotalCharges` is heavily right-skewed, with most customers below 2,000 and
  a long tail reaching 8,685.

The maximum of `TotalCharges` is consistent: 72 months x a 118.75 top monthly
fee equals 8,550, so 8,685 is plausible.

### Hypothesis: a cap on tenure

Month 72 holds 362 customers against an average of about 95 across months 63
to 69. Seventy-two months is exactly six years, which suggests the data
extraction window covers that period and that any older customer appears at
the maximum value.

It cannot be proven with the available data: cross-tabulated against contract
type, 94.8% of those customers hold a two-year contract, but that is
consistent both with censoring and with a genuine group of loyal customers.

**Decision for Phase 3:** add a `tenure_max = (tenure == 72)` flag so the model
can tell that group apart, without asserting a cause.

### Main finding: early retention

The overall churn rate is **26.5%** (1,869 out of 7,043 customers).

Broken down by tenure, a sharp pattern emerges:

| Tenure | Churn rate |
|---|---|
| 1 month | 62.0% |
| 2 months | 51.7% |
| 3 months | 47.0% |
| 4 months | 47.2% |
| 70 months | 9.2% |
| 72 months | 1.7% |

Grouping:

- Customers with **3 months or less**: 1,062 people (15.1% of the base),
  churning at **56.2%**.
- Customers with **more than 3 months**: 5,981 people, churning at **21.3%**.
- **31.9% of all churn is concentrated in that 15.1% of customers.**

**Business reading.** This company's problem is not acquiring customers but
keeping them through the first quarter. Almost a third of all churn happens
among customers with less than four months, and more than half of new
customers leave before completing the quarter. Risk falls steadily with
tenure: customers past the two-year mark barely churn at all.

### Pending

- Analyse the bimodality of `MonthlyCharges`. Hypothesis: the two groups
  correspond to customers without internet versus fibre customers.
- Check the relationship between `TotalCharges` and the product
  `tenure x MonthlyCharges`.
- Decide whether `TotalCharges` needs a transformation. Model-dependent: trees
  are indifferent, linear and distance-based models are not.

### 4.2 MonthlyCharges

In [0]:
plt.hist(pdf["MonthlyCharges"], bins=20)
plt.title("Distribución de los cargos mensuales")
plt.xlabel("Cargos mensuales")
plt.ylabel("Clientes");

In [0]:
print(pdf["MonthlyCharges"].min(), pdf["MonthlyCharges"].max())

SPA: Los clientes que están pagando menos de 20€ por mes están contratando la tarifa más barata. Posiblemente (esto es suposición) una tarifa de solo línea telefónica. Hay que comprobarlo, pues las columnas permiten estudiar estos datos.  

ENG: Customers paying less than 20€ per month are probably using the cheapest plan. Maybe (this is a guess) a plan with only phone service. We should check it, because the columns allow us to study these data.

In [0]:
pdf.groupby("InternetService")["MonthlyCharges"].describe()

SPA: Dados estos datos, que 1526 clientes pertenecen a la categoría de no haber contratado Fibra Óptica, podemos deducir de dónde viene ese gran número de clientes con tarifas bajas. Ahora bien, deberíamos estudiar si los clientes que han contratado tarifas baratas son los que más se van, si están en ese 62% del primer mes.  

ENG: Given these data, 1526 customers belong to the category of not having contracted Fiber Optic, we can deduce from where comes that large number of customers with low charges. Now, we should study if the customers who have contracted low charges are the ones who leave the most, if they are in that 62% of the first month.

In [0]:
display(pdf.groupby("InternetService")["baja"].mean())
display(pdf.groupby("InternetService")["tenure"].median())

SPA: De esta celda anterior deducimos que de los que pagan aproximadamente 20 al mes se van el 7,4%. Los que pagan aproximadamente 91, se van el 41,9%. **Los clientes más baratos parecen ser los más fieles**.  

ENG: From the previous cell we deduced that of those who pay about 20 a month they leave 7.4%. Those who pay about 91, they leave 41.9%. The cheapest customers seem to be the most loyal.

In [0]:
print(f"{pdf.loc[pdf['InternetService'] == 'Fiber optic', 'baja'].sum() / pdf['baja'].sum():.1%} "
      f"de todas las bajas son clientes de fibra óptica contratada")

### SPA — El churn (el abandono) se concentra en la fibra óptica

La tasa de baja varía enormemente según el tipo de conexión contratada:

| Servicio | Clientes | Tasa de baja |
|---|---|---|
| Fibra óptica | 3.096 | 41,9 % |
| DSL | 2.421 | 19,0 % |
| Sin internet | 1.526 | 7,4 % |

Frente a una tasa global del 26,5 %, un cliente de fibra tiene casi seis veces
más probabilidad de irse que uno con solo línea telefónica. **El 69,4 % de
todas las bajas son clientes de fibra**, pese a representar solo el 44 % de la
base.

Descartada la antigüedad como explicación: las medianas de `tenure` son
similares en los tres grupos (29, 30 y 25 meses), así que el efecto no se debe
a que los clientes de fibra sean más nuevos.

Y en sentido contrario: los clientes más baratos (unos 21 al mes) son los más
fieles, y los más caros (unos 91) los que más se marchan.

**Pregunta abierta.** Los datos no dicen por qué. Puede ser precio, calidad del
servicio o presión de la competencia sobre ese segmento. Responderlo exige
información que no está en este dataset.

### ENG — Churn is concentrated in fibre optic

Churn rate varies sharply by type of internet connection:

| Service | Customers | Churn rate |
|---|---|---|
| Fiber optic | 3,096 | 41.9% |
| DSL | 2,421 | 19.0% |
| No internet | 1,526 | 7.4% |

Against an overall rate of 26.5%, a fibre customer is almost six times more
likely to leave than a phone-only one. **69.4% of all churn comes from fibre
customers**, even though they are only 44% of the base.

Tenure is ruled out as an explanation: median `tenure` is similar across the
three groups (29, 30 and 25 months), so the effect is not caused by fibre
customers being newer.

The relationship runs the opposite way to intuition: the cheapest customers
(around 21 a month) are the most loyal, and the most expensive (around 91) the
most likely to leave.

**Open question.** The data does not say why. It could be price, service
quality or competitive pressure on that segment. Answering it requires
information not present in this dataset.

In [0]:
servicios = ["PhoneService", "MultipleLines", "InternetService", "OnlineSecurity",
             "OnlineBackup", "DeviceProtection", "TechSupport",
             "StreamingTV", "StreamingMovies"]

pdf.groupby(servicios + ["Churn"])["MonthlyCharges"].mean().unstack().dropna().head(15)

### SPA — Comprobación de fuga temporal en MonthlyCharges

`MonthlyCharges` es el importe facturado el último mes. Si a los clientes que
se dan de baja se les prorratease esa última factura, la variable estaría
contaminada por el desenlace que queremos predecir.

Para comprobarlo se agrupó por las nueve columnas de servicios contratados y,
dentro de cada paquete idéntico, se comparó el importe medio de quienes se
fueron frente al de quienes se quedaron. Dos clientes con los mismos servicios
deberían pagar lo mismo.

Las diferencias son de céntimos y se producen en ambos sentidos, no
sistemáticamente a la baja. **No hay prorrateo.** La variable se conoce antes
del desenlace y puede usarse sin riesgo de fuga.

### ENG — Temporal leakage check on MonthlyCharges

`MonthlyCharges` is the amount billed in the last month. If churning customers
had that final bill prorated, the variable would be contaminated by the very
outcome we are trying to predict.

To test this, the data was grouped by the nine service columns and, within each
identical service bundle, the average charge of those who left was compared
against those who stayed. Two customers with the same services should pay the
same.

Differences are a matter of cents and go in both directions, not systematically
downwards. **There is no proration.** The variable is known before the outcome
and can be used without leakage risk.

### 4.3 TotalCharges

SPA: Por descartar, vamos a comprobar algo tan sencillo como "un cliente que lleve 5 meses pagando 100 al mes, su TotalCharges debería ser de 500".  

ENG: Just to discard, let's check something as simple as "a customer paying 100 per month for 5 months, his TotalCharges should be 500".

In [0]:
total = pd.to_numeric(pdf["TotalCharges"], errors="coerce").fillna(0)
esperado = pdf["tenure"] * pdf["MonthlyCharges"]

(total - esperado).describe()

SPA: Era obvio que podían salir unos datos como estos. No todos los meses han tenido la misma tarifa ciertos clientes. MonthlyCharges es lo que está abonando ahora, no todo lo que ha pagado en su historial.  

ENG: It was obvious that these kind of data could come out. Not all months have had the same charge for some customers. MonthlyCharges is what they are paying now, not everything they have paid in their history.

In [0]:
pd.DataFrame({
    "tenure": pdf["tenure"],
    "MonthlyCharges": pdf["MonthlyCharges"],
    "TotalCharges": total,
    "producto": esperado,
}).corr()

In [0]:
# SPA: Comparamos frente a Churn.
# ENG: We compare against Churn.

In [0]:
# Partimos en quintiles. Cinco grupos del mismo tamaño ordenados por TotalCharges.
# ENG: We split in quintiles. Five groups of the same size ordered by TotalCharges.

pdf.groupby(pd.qcut(total, 5))["baja"].mean().plot.bar()
plt.axhline(tasa_global, color="red", linestyle="--", label=f"Media: {tasa_global:.1%}")
plt.title("Tasa de baja por quintil de facturación total")
plt.xticks(rotation=45, ha="right")
plt.legend();

In [0]:
# SPA: MonthlyCharges frente a Churn
# ENG: MonthlyCharges vs Churn

pdf.groupby(pd.qcut(pdf["MonthlyCharges"], 5))["baja"].mean().plot.bar()
plt.axhline(tasa_global, color="red", linestyle="--")
plt.title("Tasa de baja por quintil de cuota mensual")
plt.xticks(rotation=45, ha="right");

In [0]:
# SPA: Existe, además, un problema con TotalCharges que ya se venía anticipando desde la matriz de correlación. Se parece demasiado al Churn. Pero como comprobamos, TotalCharges no es una columna que sea producto de la multiplicación entre tenure y MonthlyCharges porque en diferentes meses se ha cobrado a diferentes clientes unas diferentes cantidades. Hay un margen de error que no podemos obviar. Y eso es lo que vamos a calcular ahora para ver qué tan útil y cuánto se solapa TotalCharges con Churn.
# ENG: There is, in addition, a problem with TotalCharges that has already been hinted at from the correlation matrix. It looks very much like Churn. But as we have seen, TotalCharges is not a column that is the product of the multiplication between tenure and MonthlyCharges because different amounts have been charged to different customers in different months. There is a margin of error that we cannot ignore. And that is what we are going to calculate now to see how useful and how much it overlaps with Churn.

### Conclusiones de estas gráficas:
SPA: Como ya sabíamos, las bajas se concentran sobre todo en las tarifas altas. Posibles hipótesis: Fibra de mal servicio/no relacionada con su calidad precio, o bien baja competitividad frente a otras empresas en precios.  
ENG: As expected, churn is concentrated in the high charges quintile. Possible hypothesis: Fiber optic service is not related to its quality price, or low competitiveness against other companies in terms of price.

**SPA:** Existe, además, un problema con TotalCharges que ya se venía anticipando desde la matriz de correlación. Se parece demasiado al Churn. Pero como comprobamos, TotalCharges no es una columna que sea producto de la multiplicación entre tenure y MonthlyCharges porque en diferentes meses se ha cobrado a diferentes clientes unas diferentes cantidades. Hay un margen de error que no podemos obviar. Y eso es lo que vamos a ver.

**ENG:** There is, in addition, a problem with TotalCharges that has already been hinted at from the correlation matrix. It looks very much like Churn. But as we have seen, TotalCharges is not a column that is the product of the multiplication between tenure and MonthlyCharges because different amounts have been charged to different customers in different months. There is a margin of error that we cannot ignore. And that is what we are going to see.

In [0]:
sns.barplot(x=pd.qcut(total, 5), y=pdf["baja"])
plt.axhline(tasa_global, color="red", linestyle="--")
plt.ylabel("Tasa de baja")
plt.xticks(rotation=45, ha="right");

In [0]:
sns.barplot(x=pd.qcut(pdf["MonthlyCharges"], 5), y=pdf["baja"])
plt.axhline(tasa_global, color="red", linestyle="--")
plt.ylabel("Tasa de baja")
plt.xticks(rotation=45, ha="right");

SPA: De esto (que debería haberlo hecho antes) sacamos que, en realidad, hay cuatro grupos en TotalCharges (ya que hay dos quintiles que se solapan). Pasa lo mismo con MonthlyCharges, que dos de ellos se solapan, dejando así 4 grupos reales. Esto hace que de cara al futuro planteemos un buen modelo basado en árboles de decisión. Podrían ser varias regresiones lineales separadas por grupos, pero mejor árboles de decisión.  <

ENG: From this (which should have been done before) we conclude that, actually, there are four groups in TotalCharges (because there are two overlapping quintiles). The same happens with MonthlyCharges, which two of them overlap, leaving four real groups. This makes us think ahead of a good model based on decision trees. They could be several linear regressions separated by groups, but better decision trees.

In [0]:
# SPA: Comprobamos, para cada quintil de cuota, qué porcentaje tiene DSL, fibra o nada.
# ENG: We check, for each quintile of charges, what percentage has DSL, fiber or nothing.


(pdf.groupby(pd.qcut(pdf["MonthlyCharges"], 5), observed=True)["InternetService"]
    .value_counts(normalize=True)
    .unstack())

SPA: De esto volvemos a sacar que "quien paga más al mes, se va más", pero, casi siempre, eso significa tener fibra.
Dos opciones:  
1. El precio hace que la gente se vaya y la fibra es cara.
2. La fibra hace que la gente se vaya.
Problema que podemos comprobar con la variable de confusión para ver qué puede estar ocurriendo.  

ENG: From this we again conclude that "who pays more per month, leaves more", but, almost always, that means having fiber.  
Two options:  
1. The price makes people leave and fiber is expensive.  
2. Fiber makes people leave.  
Problem that we can check with the confusion matrix to see what might be happening.  

In [0]:
fibra = pdf[pdf["InternetService"] == "Fiber optic"]

sns.barplot(x=pd.qcut(fibra["MonthlyCharges"], 4), y=fibra["baja"])
plt.axhline(fibra["baja"].mean(), color="red", linestyle="--")
plt.title("Tasa de baja por cuota — solo clientes de fibra")
plt.ylabel("Tasa de baja")
plt.xticks(rotation=45, ha="right");

**SPA:** Según datos totales: Cuanto más pagas al mes, más probable es que te vayas. Sin embargo, si miramos la fibra aislada (factor que hace que pagues más), cuanto más pagas al mes, MENOS te vas. Esta es la famosísima paradoja de Simpson. Al estratificar, la tendencia se invierte.
Es un problema que, de no haberlo visto, podríamos haberle dicho a negocio que "los clientes más caros son los que más se van" y ellos haber respondido "entonces bajamos los precios" cuando ese no era nuestro verdadero problema.
**El riesgo de abandono se concentra en las contrataciones de solo fibra. Los que pagan entre 68 y 80.**

**ENG:** According to total data: The more you pay per month, the more likely you are to leave. However, if we look at fiber alone (factor that makes you pay more), the more you pay per month, LESS you leave. This is the famous Simpson's paradox. When stratified, the trend is reversed.
**The risk of abandonment is concentrated in the contracts of only fiber. Those who pay between 68 and 80.**

In [0]:
# Comprobamos la hipótesis anterior:
# ENG: We check the previous hypothesis:


extras = ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
          "TechSupport", "StreamingTV", "StreamingMovies"]

pdf["n_servicios"] = (pdf[extras] == "Yes").sum(axis=1)

In [0]:
fibra = pdf[pdf["InternetService"] == "Fiber optic"]

sns.barplot(x=fibra["n_servicios"], y=fibra["baja"])
plt.axhline(fibra["baja"].mean(), color="red", linestyle="--")
plt.title("Tasa de baja según número de servicios contratados — fibra")
plt.ylabel("Tasa de baja");

**SPA:** Con estos datos podemos deducir dos hipótesis sobre lo que ocurre y planteárselo a negocio:
1. La gente se queda porque es un "incordio" cambiarse a otra operadora con varios servicios.
2. El cliente satisfecho y con intención de quedarse es el que va sumando extras.
Sea como sea, no podemos confirmar ninguna de las hipótesis con los datos que tenemos en mano. Ellos, sin embargo, sí podrían.  

**ENG:** With this data we can deduce two hypotheses about what is happening and ask them to business:
1. The people stay because it is a "pain" to change to another operator with several services.
2. The satisfied customer and with the intention of staying is the one who adds more extras.
Whatever the case, we cannot confirm any of the hypotheses with the data we have at hand. They, however, could.

## CONCLUSION:
### SPA — La tendencia se invierte dentro de la fibra

Mirando a todos los clientes, cuanto mayor es la cuota mensual mayor es la tasa de baja (del 9 % en el quintil más barato al 36 % en el más caro).

Al mirar **solo a los clientes de fibra**, la tendencia se da la vuelta: los que pagan entre 68 y 80 se dan de baja el 55 %, y los que pagan más de 101 solo el
26 %, por debajo incluso de la media de la empresa.  

Es un caso de paradoja de Simpson: la relación aparente entre cuota y baja estaba explicada por el tipo de conexión, no por el precio.  

**Qué la explica.** Contando cuántos de los seis servicios adicionales tiene contratado cada cliente, la relación es clara y monótona dentro de la fibra:

| Servicios contratados | Tasa de baja |
|---|---|
| 0 | 60 % |
| 1 | 55 % |
| 2 | 48 % |
| 3 | 38 % |
| 4 | 32 % |
| 5 | 19 % |
| 6 | 9 % |

El problema no es la fibra en sí, sino **la fibra contratada sola**. Un cliente de fibra con los seis complementos se da de baja menos que la media de la
empresa.  

**Consecuencias:**
- Se crea la variable derivada `n_servicios` (0 a 6) para la Fase 3. No existe en el dataset original.
- Es el primer hallazgo sobre el que el negocio puede actuar: la antigüedad y el tipo de conexión no se pueden cambiar, el número de servicios sí.
- Sobre la causalidad, ver las dos hipótesis planteadas más arriba.


### ENG — The trend reverses within fibre

Across all customers, the higher the monthly charge the higher the churn rate (from 9% in the cheapest quintile to 36% in the most expensive).

Looking at **fibre customers only**, the trend flips: those paying between 68 and 80 churn at 55%, while those paying above 101 churn at just 26%, below
the company average.  

This is a case of Simpson's paradox: the apparent link between price and churn was explained by the type of connection, not by the price itself.  

**What explains it.** Counting how many of the six add-on services each customer holds, the relationship within fibre is clear and monotonic:

| Add-on services | Churn rate |
|---|---|
| 0 | 60% |
| 1 | 55% |
| 2 | 48% |
| 3 | 38% |
| 4 | 32% |
| 5 | 19% |
| 6 | 9% |

The problem is not fibre itself but **fibre bought on its own**. A fibre customer holding all six add-ons churns less than the company average.  

**Consequences:**
- A derived feature `n_servicios` (0 to 6) is created for Phase 3. It does not exist in the original dataset.
- This is the first actionable finding: tenure and connection type cannot be changed, the number of services can.
- On causality, see the two hypotheses stated above.